# 01 - Data Loading & Graph Construction

**Mục tiêu của notebook này:**
- Load + làm sạch `USvideos.csv` qua `src/loader.py`
- Xây weighted graph qua `src/graph_builder.py`
- Export graph để dùng ở notebook `02_analysis.ipynb` và `03_models.ipynb`

> Toàn bộ logic xử lý nằm trong `src/`, notebook này chỉ gọi lại để dễ tái sử dụng ở `main.py`.


In [ ]:
import sys
from pathlib import Path

# Thêm thư mục gốc dự án vào sys.path để import được package src/
PROJECT_ROOT = Path("..").resolve()
sys.path.append(str(PROJECT_ROOT))

from src.loader import load_and_clean, save_cleaned
from src.graph_builder import build_graph, save_graph
import networkx as nx

DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "output"
(OUTPUT_DIR / "graphs").mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "tables").mkdir(parents=True, exist_ok=True)

COUNTRY = "US"  # đổi thành "GB" khi chạy cho dataset Anh


## 1-2. Load & làm sạch dữ liệu

(toàn bộ logic nằm trong `src/loader.py`)

In [ ]:
df = load_and_clean(COUNTRY, data_dir=DATA_DIR)
print(f"Số dòng sau khi làm sạch: {len(df):,}")
print(f"Số channel duy nhất: {df['channel_title'].nunique():,}")
df.head()


In [ ]:
save_cleaned(df, COUNTRY, OUTPUT_DIR / "tables")


## 3. Xây dựng graph

(toàn bộ logic nằm trong `src/graph_builder.py`)

> **Lưu ý phương pháp:** với mỗi `(trending_date, category_name)`, tất cả channel cùng trending sẽ tạo thành một **clique đầy đủ**. Có thể tăng `weight_threshold` để giảm bớt hiệu ứng này — nên thử vài giá trị khác nhau và so sánh kết quả ở notebook 02.

In [ ]:
G = build_graph(df, weight_threshold=1)
print(f"Số node: {G.number_of_nodes():,}")
print(f"Số edge: {G.number_of_edges():,}")


## 4. Thông tin cơ bản của graph

In [ ]:
degrees = [d for _, d in G.degree()]
print(f"Degree trung bình: {sum(degrees)/len(degrees):.2f}")
print(f"Degree lớn nhất: {max(degrees)}")
print(f"Density: {nx.density(G):.6f}")

# Cần cho average_shortest_path_length / diameter ở notebook 02
print(f"Số connected component: {nx.number_connected_components(G)}")
largest_cc = max(nx.connected_components(G), key=len)
print(f"Kích thước connected component lớn nhất: {len(largest_cc):,} / {G.number_of_nodes():,} node")


## 5. Export graph

In [ ]:
graph_path = save_graph(G, COUNTRY, OUTPUT_DIR / "graphs")
print(f"Đã lưu graph tại: {graph_path}")


## Tiếp theo

Dùng `src.graph_builder.load_graph(COUNTRY, OUTPUT_DIR / "graphs")` ở notebook `02_analysis.ipynb` để tính các network metrics (degree distribution, power-law, centrality, community detection).